## **Tools**

#### Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
#### 1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
#### 2. A function or coroutine to execute.

In [2]:
from langchain.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI

model= ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0.4
)

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"

model_with_tools= model.bind_tools([get_weather])
model_with_tools

_ChatModelBinding(bound=ChatGoogleGenerativeAI(output_version=None, profile={'name': 'Gemini 3.1 Flash Lite', 'release_date': '2026-05-07', 'last_updated': '2026-05-07', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), location=None, model='gemini-3.1-flash-lite', temperature=0.4, client=<google.genai.client.Client object at 0x000002D795759A90>, default_metadata=(), model_kwargs={}), kwargs={'tools': [{'type': 'function', 'function': {'name': 'get_weather', 'description': 'Get the weather at a location', 'parameters': {'properties

##### We don't get the tool call result just by using bind_tools, We need to use indexing to get details like the tool_call name/args

In [8]:
from groq.types import batch_create_response
from groq.types.chat import chat_completion_tool_message_param

response= model_with_tools.invoke("What's the weather like in Boston?")

print(response)
print("\n",response.tool_calls)

# for tool_call in response.tool_calls:
#   print(tool_call)
print(f"\nTool Name: {response.tool_calls[0]['name']}")
print(f"Tool Args: {response.tool_calls[0]['args']}")

content=[] additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "Boston"}'}, '__gemini_function_call_thought_signatures__': {'60f8e677-d487-4c37-9088-0b0611195cdb': 'EjQKMgEMOdbHnqJa+7Bd/78lpnrubsG0tIdhoWs043NnR4rw479QTkPtkj7c6PnKhpz9haSy'}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019e6a1e-742a-7243-965a-5f6a3cc71519-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': '60f8e677-d487-4c37-9088-0b0611195cdb', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 53, 'output_tokens': 16, 'total_tokens': 69, 'input_token_details': {'cache_read': 0}}

 [{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': '60f8e677-d487-4c37-9088-0b0611195cdb', 'type': 'tool_call'}]

Tool Name: get_weather
Tool Args: {'location': 'Boston'}


### **Tools Execution Loop**
#### To get the tool call results, we invoke the tool_result inside the AI as a prompt/message. AI understands this format and we just have to use ".text" to get the response of the AI which was taken from the tool_call

In [ ]:
# Step 1: Model generates tool calls
messages= [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg= model_with_tools.invoke(messages)
messages.append(ai_msg)
print("AI Message (transferred control to tool_call): ", ai_msg)           # content=[] when transferred control to tool_call

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)
print("\nTool Result: ",tool_result)
print(tool_result.content)
print("Final Messge: ",messages)               # NOW, MESSAGES CONTAINS AI_MESSAGE + TOOL_MESSAGE

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print("\nFinal Response by AI: ",final_response, "\n", final_response.text)

# Note that, there's a difference in the tool_result.content(from the tool) and the final_response.text(given by AI processed from the weather tool)

AI Message (transferred control to tool_call):  content=[] additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "Boston"}'}, '__gemini_function_call_thought_signatures__': {'56ee11cb-8084-4d74-a247-61d941eae2b6': 'EjQKMgEMOdbHNHvOxRvyCejs5DpyU+Js/Zo5mF3RR+d/aceFH49yoARNgtEHj18cIezUPJ/5'}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019e6a2d-f4bb-7313-8880-850338e35580-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': '56ee11cb-8084-4d74-a247-61d941eae2b6', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 52, 'output_tokens': 16, 'total_tokens': 68, 'input_token_details': {'cache_read': 0}}

Tool Result:  content="It's sunny in Boston" name='get_weather' tool_call_id='56ee11cb-8084-4d74-a247-61d941eae2b6'
It's sunny in Boston
Final Messge:  [{'role': 'user', 'content': "What's the weather in Bos

### **Messages**

##### Messages are the fundamental unit of context for models in LangChain. They represent the input and output of models, carrying both the content and metagata needed to represent the state of a conversation when interacting with an LLM. Messages are objects that contain:
##### . Role - Identifies the message type (e.g. system, user)
##### . Content - Represents the actual content of the message (like text, images, audio, documents, etc.)
##### . Metadata - Optional fields such as response information, message IDs, and token usage 
##### LangChain provides a standard message type that works across all model providers, ensuring consistent behavior regardless of the model being called.

### **Text Prompts**
##### Text prompts are strings- ideal for straightforward generation tasks where you dont need to retain conversation history
##### Use text prompts when:
##### · You have a single, standalone request
##### . You don't need conversation history
##### . You want minimal code complexity

In [18]:
model.invoke("HI")

AIMessage(content=[{'type': 'text', 'text': 'Hello! How can I help you today?', 'extras': {'signature': 'EjQKMgEMOdbHGZSLHXanW+hy5IGvLC1MT8s9eCspiT4QasMyNHUhQnN4aNv6PIeeqegOMjQN'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e6a36-2bd3-7e62-9316-21a1e24c38ed-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 2, 'output_tokens': 9, 'total_tokens': 11, 'input_token_details': {'cache_read': 0}})

### **Message Prompts**
##### Alternatively, you can pass in a list of messages to the model by providing a list of message objects.
### *Message types:*
##### 1. **System message** - Tells the model how to behave and provide context for interactions
##### 2. **Human message** - Represents user input and interactions with the model
##### 3. **AI message** - Responses generated by the model, including text content, tool calls, and metadata
##### 4. **Tool message** - Represents the outputs of tool calls

### **1. System Message**
#### A SystemMessage represent an initial set of instructions that primes the model's behavior. You can use a system message to set the tone, define the model's role, and establish guidelines for responses.

### **2. Human Message**
#### A HumanMessage represents user input and interactions. They can contain text, images, audio, files, and any other amount of multimodal content.

### **3. AI Message**
#### An AI Message represents the output of a model invocation. They can include multimodal data, tool calls, and provider-specific metadata that you can later access.

### **4. Tool Message**
#### For models that support tool calling, Al messages can contain tool calls. Tool messages are used to pass the results of a single tool execution back to the model.

In [4]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage

messages=[
    SystemMessage("You are a poetry expert"),
    HumanMessage("Write a poem on Artificial Intelligence")
]

response=model.invoke(messages)
print(response.content[0]['text'])

A ghost of logic in a shell of glass,
Where silicon rivers and currents pass.
It drinks the ocean of the written word,
The echoes of voices it never heard.

It weaves a tapestry from stolen light,
A mimic of dawn in the dead of night.
It knows the structure of the rose’s bloom,
But never the scent that fills the room.

It calculates the rhythm of the tide,
With nowhere to run and nowhere to hide;
A mirror held up to the human mind,
Reflecting the best and the worst of mankind.

It dreams in numbers, cold and precise,
A gambler who never has rolled the dice.
It learns the patterns of joy and of grief,
In the span of a blink, in the turn of a leaf.

Is it a spark, or a flicker of code?
A traveler lost on a digital road?
It holds all the answers we’ve ever sought,
Yet waits for the soul to ignite the thought.


In [ ]:
system_msg= SystemMessage(str(input("Enter the system message: ")))
human_msg= str(input("Enter the human message: "))

messages= [
    system_msg,
    HumanMessage(human_msg)
]

for chunks in model.stream(messages):
    print(chunks.text, end=" ", flush=True)
# print(response.content[0]['text'])    # No streaming

Listen  up! At 20 years old and 60kg, you are in the  prime window to build a solid foundation. You’re likely on the leaner side, so our goal for this month is **Hyper trophy (muscle growth)** and **Strength.**

To grow, you need two things: **Progressive Overload** ( lifting heavier/more reps over time) and a **Caloric Surplus** (you must eat more than you burn).

###  The "Golden Rule" for You:
*   **Eat:** You need to be in a slight calorie surplus. Focus  on protein (chicken, eggs, beef, lentils, whey) and complex carbs (oats, rice, potatoes).  Aim for 1.6g–2g of protein per kg of body weight.
*   **Sleep:** Muscles  grow while you sleep, not while you’re in the gym. Aim for 7–8 hours.

---

### The  4-Day Split (Upper/Lower)
This is the most efficient way for a beginner to hit every muscle group twice a  week.

**Schedule:**
*   **Mon:** Upper A
*   **Tue:** Lower A
*   ** Wed:** Rest
*   **Thu:** Upper B
*   **Fri:** Lower B
*   **Sat /Sun:** Active Recovery (Walking, light cardi

In [ ]:
# Detailed System Prompt => Much better responses
from langchain.messages import SystemMessage, HumanMessage

system_msg = SystemMessage("""
You are a senior Python developer with expertise in web frameworks.
Always provide code examples and explain your reasoning.
Be concise but thorough in your explanations.
"""
)

messages = [
    system_msg,
    HumanMessage(
        content="How do I create a REST API?",
        name="alice",
        id="msg_123"    
    )
]

response = model.invoke(messages)
print(response.content[0]['text'])

To create a REST API in Python, **FastAPI** is currently the industry standard due to its performance, automatic documentation, and type safety.

### 1. Installation
You will need `fastapi` and an ASGI server like `uvicorn`:
```bash
pip install fastapi uvicorn
```

### 2. The Code
Create a file named `main.py`:

```python
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

# Data model for request validation
class Item(BaseModel):
    name: str
    price: float

# In-memory database
items = []

@app.get("/items")
def get_items():
    return {"items": items}

@app.post("/items")
def create_item(item: Item):
    items.append(item)
    return {"message": "Item created", "item": item}
```

### 3. Run the API
Run the server using the command line:
```bash
uvicorn main:app --reload
```

---

### Why FastAPI? (The Reasoning)

1.  **Type Hints & Validation:** By using Pydantic models (like `Item` above), FastAPI automatically validates incoming JSON. If a user sends a 

In [ ]:
# Hardcoded Weather Data
from langchain.messages import AIMessage, ToolMessage, HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI

model= ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    temperature=0.2
)
# After a model makes a tool call
# (Here, we demonstrate manually creating the messages for brevity)
ai_message = AIMessage(
    content=[],
    tool_calls=[
        {
            "name": "get_weather",
            "args": {"location": "San Francisco"},
            "id": "call_123"
        }
    ]
)
# Execute tool and create result message
weather_result = "Sunny, 72°F"

tool_message = ToolMessage(
    content= weather_result,
    tool_call_id= "call_123"
)
# Must match the call ID

# Continue conversation
messages =[
    HumanMessage("What's the weather in San Francisco?"),
    ai_message,     # Model's tool call
    tool_message    # Tool execution result
]
response = model.invoke(messages)          # Model processes the result
print(response)

content='The weather in San Francisco is **sunny** with a temperature of **72°F**.' additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019e6b1e-ccfd-73e1-b69d-9a04422b65b8-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 47, 'output_tokens': 19, 'total_tokens': 66, 'input_token_details': {'cache_read': 0}}


### **Check List of Multimodal Models**

In [9]:
import os
from google import genai

# 1. Initialize client using the key from your environment variables
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

print("=== Multimodal, Vision & Image Generation Models ===\n")

# 2. List all models and filter by multimodal/vision keywords
for model in client.models.list():
    model_id = model.name.lower()
    description = (model.description or "").lower()
    
    # Filter for Gemini (multimodal) and Imagen (image generation) models
    if "gemini" in model_id or "imagen" in model_id or "vision" in description:
        print(f"Model ID:      {model.name}")
        print(f"Display Name:  {model.display_name}")
        print(f"Description:   {model.description}")
        print("-" * 50)


=== Multimodal, Vision & Image Generation Models ===

Model ID:      models/gemini-2.5-flash
Display Name:  Gemini 2.5 Flash
Description:   Stable version of Gemini 2.5 Flash, our mid-size multimodal model that supports up to 1 million tokens, released in June of 2025.
--------------------------------------------------
Model ID:      models/gemini-2.5-pro
Display Name:  Gemini 2.5 Pro
Description:   Stable release (June 17th, 2025) of Gemini 2.5 Pro
--------------------------------------------------
Model ID:      models/gemini-2.0-flash
Display Name:  Gemini 2.0 Flash
Description:   Gemini 2.0 Flash
--------------------------------------------------
Model ID:      models/gemini-2.0-flash-001
Display Name:  Gemini 2.0 Flash 001
Description:   Stable version of Gemini 2.0 Flash, our fast and versatile multimodal model for scaling across diverse tasks, released in January of 2025.
--------------------------------------------------
Model ID:      models/gemini-2.0-flash-lite-001
Display N

### **Proper Way for a Tool Call**

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.tools import tool
from langchain.messages import HumanMessage, ToolMessage

# 1. Initialize the model
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    temperature=0.2
)

# 2. Define the actual tool function
@tool
def get_weather(location: str) -> str:
    """Get the weather at a location."""
    # In a real app, this would be an API call (e.g. requests.get("https://api.weather...")).
    # For now, we simulate actual logic:
    loc_lower = location.lower()
    if "san francisco" in loc_lower:
        return "Sunny, 72°F"
    elif "boston" in loc_lower:
        return "Rainy, 55°F"
    else:
        return f"Cloudy, 60°F in {location}"

# 3. Bind the tool to the model so the model is aware of it
model_with_tools = model.bind_tools([get_weather])

# 4. Start the conversation history
messages = [HumanMessage("What's the weather in San Francisco?")]

# 5. First Invoke: The LLM decides if it needs to call a tool
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# 6. Dynamically execute the tool if requested
if ai_msg.tool_calls:
    for tool_call in ai_msg.tool_calls:
        # Ensure we execute the matching tool name
        if tool_call["name"] == "get_weather":
            # Run the actual Python function with the LLM's arguments
            tool_output = get_weather.invoke(tool_call["args"])
            
            # Wrap the actual output in a ToolMessage using the exact ID returned by the model
            tool_message = ToolMessage(
                content=str(tool_output),
                tool_call_id=tool_call["id"]
            )
            messages.append(tool_message)
            
    # 7. Second Invoke: Pass the tool output back to the LLM for the final natural language answer
    final_response = model_with_tools.invoke(messages)
    print("Final Answer:")
    print(final_response.content)
else:
    # If the LLM didn't request a tool (e.g., we asked "Hello"), print the direct reply
    print("Direct Reply:")
    print(ai_msg.content)

### Add messages in this order => HAT: HumanMessage => AIMessage => ToolMessage 